# ERA5-Land Annual All-Sites Run

This notebook launches the annual ERA5-Land watershed exports for all uploaded watershed sites. It exports one CSV per year to Google Drive, using the corrected polygon-first extraction with the fine-scale polygon retry for tiny watersheds.

The default run covers 2000-2025 and includes all annual ERA5-Land products currently used by the workflow: precipitation, air temperature, actual evapotranspiration, potential evapotranspiration, snow cover, and snow-water equivalent. Colab and Earth Engine may ask you to authenticate.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess
from pathlib import Path

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run(["pip", "-q", "install", "-r", "requirements.txt"], check=True)


In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")

DRIVE_EXPORT_FOLDER = "Google-Earth-Engine"

assets.setdefault("exports", {}).update(
    {
        "destination": "drive",
        "drive_folder": DRIVE_EXPORT_FOLDER,
    }
)

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed asset:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder"))


## Edit Run Settings

Change the settings in the next cell before choosing **Runtime > Run all**. Defaults run all sites, all annual products, and all years from 2000-2025. Use the filters for a quick test or a targeted rerun. The launch and polling cells keep a wall-clock timer for the whole run and write a small timing summary CSV under `timing-logs/`.


In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

# Output names include RUN_LABEL. Keep the default for the complete annual run.
RUN_LABEL = "all_sites_fine_scale"

# Years are inclusive. Use YEARS_TO_SKIP for partial reruns after some exports finish.
START_YEAR = 2000
END_YEAR = 2025
YEARS_TO_SKIP = []

# Leave as None for the full run. Set to 1 or 2 for a quick test.
MAX_TASKS_TO_LAUNCH = None

# Site options: "all_sites", "lter_list", or "site_id_list".
SITE_FILTER_MODE = "all_sites"
LTER_FILTERS = []  # Example: ["AND", "HBR"] when SITE_FILTER_MODE = "lter_list".
SITE_IDS = []  # Example: ["and__and_1", "and__and_2"] when SITE_FILTER_MODE = "site_id_list".

# Product options: precip, temp, evapotrans, potential_evap, snow_cover, snow_water_equiv.
PRODUCTS = [
    "precip",
    "temp",
    "evapotrans",
    "potential_evap",
    "snow_cover",
    "snow_water_equiv",
]

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()
site_id_column = choose_property_name(watersheds, "site_id")
lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])
lter_values = watersheds.aggregate_array(lter_column).distinct().sort().getInfo()
lter_lookup = {str(value).strip().lower(): value for value in lter_values}

if SITE_FILTER_MODE == "all_sites":
    selected_watersheds = watersheds
elif SITE_FILTER_MODE == "lter_list":
    if not LTER_FILTERS:
        raise ValueError('Set LTER_FILTERS when SITE_FILTER_MODE = "lter_list".')
    selected_lter_values = [
        lter_lookup.get(str(value).strip().lower(), value)
        for value in LTER_FILTERS
    ]
    selected_watersheds = watersheds.filter(ee.Filter.inList(lter_column, selected_lter_values))
elif SITE_FILTER_MODE == "site_id_list":
    if not SITE_IDS:
        raise ValueError('Set SITE_IDS when SITE_FILTER_MODE = "site_id_list".')
    selected_watersheds = watersheds.filter(ee.Filter.inList(site_id_column, SITE_IDS))
else:
    raise ValueError('SITE_FILTER_MODE must be "all_sites", "lter_list", or "site_id_list".')

selected_row_count = selected_watersheds.size().getInfo()

print("Asset rows:", watersheds.size().getInfo())
print("Selected watershed rows:", selected_row_count)
print("Asset columns:", property_names)
print("Site ID column:", site_id_column)
print("LTER column:", lter_column)
print("LTER values:", lter_values)
print("Run label:", RUN_LABEL)
print("Year range:", f"{START_YEAR}-{END_YEAR}")
print("Years skipped:", YEARS_TO_SKIP)
print("Site filter mode:", SITE_FILTER_MODE)
print("LTER filters:", LTER_FILTERS)
print("Site ID filters:", SITE_IDS)
print("Products:", PRODUCTS)
print("Selected preview:")
print(selected_watersheds.limit(5).getInfo())

if selected_row_count == 0:
    raise ValueError("No watershed rows were selected.")


In [ ]:
raise RuntimeError("Launch blocked: this legacy annual notebook has not been migrated to the mandatory GEE quota receipt. Use the guarded monthly notebook or migrate this launch cell before submitting tasks.")

import csv
import time
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import era5_export_columns, extract_era5_land_products
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

years_to_launch = [
    year for year in range(START_YEAR, END_YEAR + 1)
    if year not in set(YEARS_TO_SKIP)
]
if MAX_TASKS_TO_LAUNCH is not None:
    years_to_launch = years_to_launch[: int(MAX_TASKS_TO_LAUNCH)]

run_name = f"era5_land_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}"
run_wall_clock_started_at = utc_now()
run_wall_clock_timer_started = time.perf_counter()
session_started_at = run_wall_clock_started_at

timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_era5_land_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
run_timing_summary_path = Path("timing-logs") / (
    f"gee_run_wall_clock_era5_land_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)


def iso_or_blank(value):
    return value.isoformat(timespec="seconds") if value else ""


def write_run_wall_clock_summary(status, finished_at=None, elapsed_min=None):
    if elapsed_min is None and finished_at is not None:
        elapsed_min = (finished_at - run_wall_clock_started_at).total_seconds() / 60
    elapsed_hr = elapsed_min / 60 if elapsed_min is not None else ""
    row = {
        "run_name": run_name,
        "run_label": RUN_LABEL,
        "status": status,
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "years_to_launch": ";".join(str(year) for year in years_to_launch),
        "years_skipped": ";".join(str(year) for year in YEARS_TO_SKIP),
        "max_tasks_to_launch": "" if MAX_TASKS_TO_LAUNCH is None else MAX_TASKS_TO_LAUNCH,
        "n_tasks": len(years_to_launch),
        "selected_rows": selected_row_count,
        "site_filter_mode": SITE_FILTER_MODE,
        "lter_filters": ";".join(str(value) for value in LTER_FILTERS),
        "site_id_filters": ";".join(str(value) for value in SITE_IDS),
        "products": ";".join(PRODUCTS),
        "started_at_utc": iso_or_blank(run_wall_clock_started_at),
        "finished_at_utc": iso_or_blank(finished_at),
        "elapsed_min": "" if elapsed_min is None else f"{elapsed_min:.3f}",
        "elapsed_hr": "" if elapsed_hr == "" else f"{elapsed_hr:.3f}",
        "task_timing_log": str(timing_log_path),
        "drive_export_folder": assets["exports"].get("drive_folder", ""),
    }
    with run_timing_summary_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row))
        writer.writeheader()
        writer.writerow(row)
    return row


selectors = era5_export_columns(products, product_names=PRODUCTS)
launched_tasks = []

print("Run wall-clock start:", run_wall_clock_started_at.isoformat(timespec="seconds"))
print("Years to launch:", years_to_launch)
print("Years skipped:", YEARS_TO_SKIP)
print("Selected watershed rows:", selected_row_count)
print("Drive output folder:", assets["exports"].get("drive_folder"))
print("Export columns:", selectors)
print("Task timing log:", timing_log_path)
print("Run timing summary:", run_timing_summary_path)

for year in years_to_launch:
    out_name = f"era5_land_{year}_{RUN_LABEL}_watershed_extract"
    print()
    print("Launching:", out_name)

    export_rows = extract_era5_land_products(
        products,
        selected_watersheds,
        year=year,
        product_names=PRODUCTS,
    )

    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    launched_at = utc_now()
    run_row = {
        "run_name": run_name,
        "mode": "era5_land",
        "products": PRODUCTS,
        "year": year,
        "month": None,
        "months": None,
        "period": "annual",
        "run_group": RUN_LABEL,
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {
            "name": out_name,
            "task": task,
            "launched_at": launched_at,
            "timing_row": timing_row,
        }
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

write_run_wall_clock_summary(status="launched")

print()
print("Launched tasks:", len(launched_tasks))
print("Task timing log:", timing_log_path)
print("Run timing summary:", run_timing_summary_path)



In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60

if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this session.")
else:
    while True:
        now = utc_now()
        print()
        print("Task status at", now.isoformat(timespec="seconds"))
        all_done = True

        for item in launched_tasks:
            item["timing_row"] = update_task_timing_row(
                item["timing_row"],
                item["task"],
                checked_at=now,
            )
            state = item["timing_row"].get("state")
            elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")

            if state not in DONE_STATES:
                all_done = False

        write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)

        if all_done:
            break

        time.sleep(poll_seconds)

    run_wall_clock_finished_at = utc_now()
    if "run_wall_clock_timer_started" in globals():
        run_wall_clock_elapsed_min = (time.perf_counter() - run_wall_clock_timer_started) / 60
    elif "run_wall_clock_started_at" in globals():
        run_wall_clock_elapsed_min = (
            run_wall_clock_finished_at - run_wall_clock_started_at
        ).total_seconds() / 60
    else:
        run_wall_clock_elapsed_min = None

    print_timing_summary(timing_rows_from_launched_tasks(launched_tasks))
    print()
    print("Run wall-clock finish:", run_wall_clock_finished_at.isoformat(timespec="seconds"))
    if run_wall_clock_elapsed_min is None:
        print("Run wall-clock elapsed: unavailable because the start time was not found in this Colab session.")
    else:
        print(
            "Run wall-clock elapsed: "
            f"{run_wall_clock_elapsed_min:.1f} min "
            f"({run_wall_clock_elapsed_min / 60:.2f} hr)"
        )

    if "write_run_wall_clock_summary" in globals():
        write_run_wall_clock_summary(
            status="completed",
            finished_at=run_wall_clock_finished_at,
            elapsed_min=run_wall_clock_elapsed_min,
        )
        print("Run timing summary:", run_timing_summary_path)

    print("Task timing log:", timing_log_path)



## After The Exports Finish

This notebook stops after Earth Engine tasks finish and the CSVs are written to the Drive export folder named `Google-Earth-Engine`. Use the local R commands in `tools/gee_colab_helpers/README.md` to move the CSVs into a tidy run folder and run QA or inventory steps.
